# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets in the dataset and their fields
print("Available record sets and fields:\n")
record_sets = list(metadata.record_sets)
for rs in record_sets:
    print(f"RecordSet: {rs['@id']}")
    fields = rs.get('cr:field', [])
    # Ensure list
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for f in fields:
        field_id = f.get('@id', f)
        print(f"    - {field_id}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Select the main record set (update with the actual @id from above)
# Most datasets will have the main data file as the first record set; adjust as needed
record_sets_ids = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Choose the first record set for exploration
default_rs = record_sets_ids[0]
print(f"Columns in RecordSet {default_rs}:\n{dataframes[default_rs].columns.tolist()}")
dataframes[default_rs].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

In [ ]:
# Identify a numeric field from columns (update field_id as needed)
print("Numeric columns detected:")
for col in dataframes[default_rs].columns:
    if pd.api.types.is_numeric_dtype(dataframes[default_rs][col].dropna()):
        print("  -", col)
# For illustration, let's use a common field name related to protein count or value:
numeric_field = None
for try_name in ['cr:Abundance', 'cr:MolecularWeight', 'cr:Coverage', 'cr:PeptideCount', 'cr:MW', 'abundance', 'coverage', 'peptide_count', 'mw']:
    if try_name in dataframes[default_rs].columns:
        numeric_field = try_name
        break
if numeric_field is None:
    # Use a numeric column as fallback
    numeric_cols = [col for col in dataframes[default_rs].columns if pd.api.types.is_numeric_dtype(dataframes[default_rs][col].dropna())]
    if numeric_cols:
        numeric_field = numeric_cols[0]
    else:
        raise ValueError("No numeric field found for demonstration.")
print(f"\nUsing numeric field '{numeric_field}' for filtering and normalization.")

# Filtering and normalization
threshold = dataframes[default_rs][numeric_field].mean() or 10
filtered_df = dataframes[default_rs][dataframes[default_rs][numeric_field] > threshold].copy()
print(f"Filtered records with {numeric_field} > {threshold:.2f}:\n", filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a sample or protein type field
group_field = None
for try_group in ['cr:Sample', 'cr:ProteinType', 'sample', 'Sample', 'type']:
    if try_group in dataframes[default_rs].columns:
        group_field = try_group
        break
if group_field:
    grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
    print(f"\nGrouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(7, 4))
sns.histplot(dataframes[default_rs][numeric_field].dropna(), bins=30, kde=True)
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# If there are at least two numeric columns, make a scatter plot
numeric_cols = [col for col in dataframes[default_rs].columns if pd.api.types.is_numeric_dtype(dataframes[default_rs][col].dropna())]
if len(numeric_cols) > 1:
    x_col, y_col = numeric_cols[:2]
    plt.figure(figsize=(6, 5))
    sns.scatterplot(data=dataframes[default_rs], x=x_col, y=y_col)
    plt.title(f'Scatter plot of {x_col} vs. {y_col}')
    plt.show()

## 6. Conclusion
In this notebook, you have loaded the FAIR² dataset using a Croissant schema, explored its record sets and fields by their `@id`, extracted the data, performed EDA including filtering and normalization, and visualized distribution(s). For further analysis, proceed with domain-specific investigations, advanced modeling, or integration with other datasets.